# 2 · Renderizar (HyperFrames na CPU do Colab)

Renderiza uma composição de `EditorVideos/HyperFrames/composicoes/<nome>` e grava o MP4 em `EditorVideos/HyperFrames/saida`.
Usa a mesma versão do HyperFrames do PC (0.8.225), sem telemetria. Node é baixado do nodejs.org e conferido por SHA-256.

**Antes:** Ambiente de execução → Alterar tipo → **CPU com RAM alta** (não precisa de GPU). Depois: Executar tudo.
O andamento aparece no Painel, aba Colab.

In [ ]:
#@title 1. Configuração
COMPOSICAO = ""  #@param {type:"string"}
#@markdown Nome da pasta em HyperFrames/composicoes.
QUALIDADE = "looks"  #@param ["draft", "looks", "delivery"]
WORKERS = 4  #@param {type:"integer"}

# Pelo Colab CLI os valores chegam por variável de ambiente (colab exec --env).
import os
COMPOSICAO = os.environ.get('COMPOSICAO', COMPOSICAO)
QUALIDADE = os.environ.get('QUALIDADE', QUALIDADE)
WORKERS = int(os.environ.get('WORKERS', WORKERS))

In [ ]:
#@title 2. Drive e status para o Painel
from google.colab import drive
import json, os, time, threading, subprocess, collections
from datetime import datetime

RAIZ = '/content/drive/MyDrive/EditorVideos'
STATUS_DIR = f'{RAIZ}/_colab/status'

# A pasta EditorVideos vem do PC (Drive para computador). Se a montagem do Colab estiver velha ela
# não aparece; remonta e, se ainda faltar, para aqui em vez de criar uma "EditorVideos (1)" vazia.
# Pelo Colab CLI (Painel/colab-cli) o Drive já vem montado por `colab drivemount`: drive.mount() aqui
# ficaria esperando um navegador que não existe.
# Tarefas que não precisam do Drive (ex.: a voz pelo CLI) mandam EDITORVIDEOS_SEM_DRIVE=1: o status fica na VM
# e o Painel acompanha pelo log do CLI.
SEM_DRIVE = os.environ.get('EDITORVIDEOS_SEM_DRIVE') == '1'
if SEM_DRIVE:
    STATUS_DIR = '/content/_colab/status'
    print('Sem Drive nesta tarefa: o andamento vai pelo log do Colab CLI.')
elif os.environ.get('EDITORVIDEOS_CLI') and os.path.isdir('/content/drive/MyDrive'):
    print('Drive montado pelo Colab CLI.')
else:
    drive.mount('/content/drive', force_remount=True)
    if not os.path.isdir(f'{RAIZ}/HyperFrames'):
        time.sleep(10)
        drive.flush_and_unmount()
        drive.mount('/content/drive', force_remount=True)
if not SEM_DRIVE and not os.path.isdir(f'{RAIZ}/HyperFrames'):
    raise SystemExit('EditorVideos/HyperFrames não está visível neste Drive. Confira se o Colab está na mesma conta '
                     'do Drive do PC e se a sincronização terminou; depois Ambiente de execução → Desconectar e excluir.')

class Status:
    """Grava o andamento em _colab/status/<servico>.json. Batimento a cada 15 s enquanto roda."""
    def __init__(self, servico, notebook):
        os.makedirs(STATUS_DIR, exist_ok=True)
        self.arq = f'{STATUS_DIR}/{servico}.json'
        self.d = dict(servico=servico, notebook=notebook, estado='rodando', etapa='iniciando',
                      progresso=None, mensagem='', inicio=int(time.time() * 1000))
        self.log = collections.deque(maxlen=40)
        self._trava = threading.Lock()
        self._ultimo = 0
        self._vivo = True
        threading.Thread(target=self._batimento, daemon=True).start()
        self.gravar()

    def _gpu(self):
        try:
            out = subprocess.run(['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total',
                                  '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=5).stdout
            nome, uso, usada, total = [x.strip() for x in out.strip().splitlines()[0].split(',')]
            return dict(nome=nome, uso=int(uso), memoria_usada_mb=int(usada), memoria_total_mb=int(total))
        except Exception:
            return None

    def _ram(self):
        try:
            m = {}
            for linha in open('/proc/meminfo'):
                k, v = linha.split(':')
                m[k] = int(v.split()[0])
            return dict(usada_mb=(m['MemTotal'] - m['MemAvailable']) // 1024, total_mb=m['MemTotal'] // 1024)
        except Exception:
            return None

    def gravar(self):
        with self._trava:
            self.d.update(log=list(self.log), gpu=self._gpu(), ram=self._ram(), atualizado=int(time.time() * 1000))
            tmp = self.arq + '.tmp'
            with open(tmp, 'w') as f:
                json.dump(self.d, f, ensure_ascii=False)
            os.replace(tmp, self.arq)
            self._ultimo = time.time()

    def _batimento(self):
        while self._vivo:
            time.sleep(15)
            if self._vivo:
                self.gravar()

    def __call__(self, etapa=None, progresso=None, mensagem=None, linha=None):
        mudou = etapa is not None and etapa != self.d['etapa']
        if etapa is not None: self.d['etapa'] = etapa
        if progresso is not None: self.d['progresso'] = round(progresso, 1)
        if mensagem is not None: self.d['mensagem'] = mensagem
        if linha:
            self.log.append(f'{datetime.now():%H:%M:%S} {linha}')
            print(linha)
        # Não martela o Drive: no máximo uma gravação a cada 3 s, salvo mudança de etapa.
        if mudou or linha or time.time() - self._ultimo > 3:
            self.gravar()

    def fim(self, estado='concluido', mensagem=''):
        self._vivo = False
        self.d.update(estado=estado, mensagem=mensagem, etapa='fim' if estado == 'concluido' else self.d['etapa'])
        if estado == 'concluido': self.d['progresso'] = 100
        self.gravar()

print('Drive montado. Status em', STATUS_DIR)

In [ ]:
#@title 3. Instalar Node 22 + HyperFrames 0.8.225
NODE = 'v22.20.0'
st = Status('render', '2_renderizar')
st('instalando', 2, f'Node {NODE} + hyperframes 0.8.225', 'baixando Node')
!cd /content && curl -fsSLO https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz && curl -fsSLO https://nodejs.org/dist/{NODE}/SHASUMS256.txt
!cd /content && grep " node-{NODE}-linux-x64.tar.xz$" SHASUMS256.txt | sha256sum -c - && tar -xJf node-{NODE}-linux-x64.tar.xz
os.environ['PATH'] = f'/content/node-{NODE}-linux-x64/bin:' + os.environ['PATH']
os.environ.update(HYPERFRAMES_NO_TELEMETRY='1', DO_NOT_TRACK='1', HYPERFRAMES_SKIP_SKILLS='1')
# Emoji colorido dos adesivos/legendas: sem esta fonte o Chrome do Colab desenha quadradinhos.
!apt-get -qq install -y fonts-noto-color-emoji > /dev/null

os.makedirs('/content/hf', exist_ok=True)
with open('/content/hf/package.json', 'w') as f:
    json.dump({'private': True, 'dependencies': {'hyperframes': '0.8.225'}, 'overrides': {'undici': '7.30.1'}}, f)
st(linha='npm install hyperframes')
!cd /content/hf && npm install --no-audit --no-fund --loglevel=error
st(linha='baixando o Chrome do render')
!cd /content/hf && node node_modules/hyperframes/bin/hyperframes.mjs telemetry disable; node node_modules/hyperframes/bin/hyperframes.mjs browser ensure
st('instalado', 10, linha='pronto para renderizar')

In [ ]:
#@title 4. Renderizar
import re, shutil, pathlib
HF = '/content/hf/node_modules/hyperframes/bin/hyperframes.mjs'
ORIGEM = f'{RAIZ}/HyperFrames/composicoes/{COMPOSICAO}'
SAIDA = f'{RAIZ}/HyperFrames/saida'
try:
    if not COMPOSICAO or not os.path.exists(f'{ORIGEM}/index.html'):
        raise FileNotFoundError(f'Composição não encontrada: {ORIGEM}')
    # Copia para o disco local do Colab: o render lê muito arquivo e o Drive montado é lento.
    local = f'/content/comp/{COMPOSICAO}'
    shutil.rmtree(local, ignore_errors=True)
    shutil.copytree(ORIGEM, local, ignore=shutil.ignore_patterns('.hyperframes', 'renders', 'node_modules'))
    nome = f'{COMPOSICAO}_{time.strftime("%Y%m%d-%H%M")}_colab.mp4'
    st(f'renderizando ({QUALIDADE})', 10, COMPOSICAO, f'render {COMPOSICAO} · {QUALIDADE} · {WORKERS} workers')
    proc = subprocess.Popen(['node', HF, 'render', local, '-o', f'/content/{nome}', '-q', QUALIDADE, '-w', str(WORKERS)],
                            cwd=local, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linha in proc.stdout:
        linha = re.sub(r'\x1b\[[0-9;?]*[A-Za-z]', '', linha).strip()
        if not linha: continue
        pct = re.search(r'(\d{1,3}(?:\.\d+)?)\s?%', linha)
        if pct and float(pct.group(1)) <= 100:
            st(progresso=10 + float(pct.group(1)) * 0.85)
        else:
            st(linha=linha[:200])
    if proc.wait() != 0:
        raise RuntimeError(f'render saiu com código {proc.returncode}')
    # A CLI pode sair com 0 mesmo com "Render failed" (ex.: mixagem de áudio bloqueada): confere o arquivo.
    if not os.path.exists(f'/content/{nome}'):
        raise RuntimeError('o render não gerou o MP4; veja o log acima')
    st('copiando para o Drive', 96, linha=f'copiando {nome}')
    os.makedirs(SAIDA, exist_ok=True)
    shutil.copy(f'/content/{nome}', f'{SAIDA}/{nome}')
    st.fim('concluido', f'HyperFrames/saida/{nome}')
except Exception as e:
    st(linha=f'ERRO: {e}')
    st.fim('erro', str(e))
    raise